
## 04 - Feature Engineering

### Business Objective

The objective of this notebook is to transform
raw credit card data into meaningful analytical
features for credit risk modeling.

### Scope

- Create repayment behavior indicators
- Analyze historical delinquency
- Calculate credit utilization
- Analyze payment-to-bill ratios
- Create customer risk segments
- Prepare modeling dataset

### Modeling Target

DEFAULT - default payment next month.


In [9]:
# Technical setup: import libraries for data wrangling, visualisation and feature engineering.

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42


In [10]:
# Technical setup: define the project folders so the notebook runs from the repo root or notebooks directory.

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
FIGURES_PATH = PROJECT_ROOT / "reports" / "figures"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)


In [11]:
# Business context: load the raw credit-card dataset that will be transformed into model-ready features.

df = pd.read_csv(
    RAW_PATH / "credit_card_default.csv"
)

print("Dataset shape:", df.shape)

display(df.head())


Dataset shape: (30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


### 1. Data validation

Before creating engineered features, we verify that the business target and the core attributes are present, valid, and ready for modeling.


In [12]:
# Technical safeguard: validate the target column and the most important business fields before modifying the dataset.

assert "DEFAULT" in df.columns
assert "LIMIT_BAL" in df.columns
assert df["DEFAULT"].isin([0, 1]).all()

print("Data validation: OK")


Data validation: OK


### 2. Working dataset setup

We keep the raw data untouched and work on a copied dataframe so feature experiments remain traceable and reversible.


In [13]:
# Technical rule: protect the raw source table and build all engineered features on a working copy.

df_fe = df.copy()


### 3. Payment behavior features

The PAY_* indicators represent recent repayment status and are one of the strongest behavioral signals for credit-risk analysis.
A customer with repeated late payments is usually more likely to default in the next cycle.


In [14]:
# Technical review: inspect recent repayment behaviour before building derived delinquency features.

payment_columns = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

payment_columns = [
    col for col in payment_columns
    if col in df_fe.columns
]

display(
    df_fe[payment_columns]
    .describe()
)


,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
count,"30,000.000","30,000.000","30,000.000","30,000.000","30,000.000","30,000.000"
mean,-0.017,-0.134,-0.166,-0.221,-0.266,-0.291
std,1.124,1.197,1.197,1.169,1.133,1.150
min,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000
25%,-1.000,-1.000,-1.000,-1.000,-1.000,-1.000
50%,0.000,0.000,0.000,0.000,0.000,0.000
75%,0.000,0.000,0.000,0.000,0.000,0.000
max,8.000,8.000,8.000,8.000,8.000,8.000


In [15]:
# Business signal: create a summary flag indicating whether the customer was late in any recent payment period.

df_fe["ANY_PAST_DUE"] = (
    df_fe[payment_columns]
    .gt(0)
    .any(axis=1)
    .astype(int)
)


In [16]:
# Business reporting: summarise the share of customers with delayed payments to understand portfolio risk.

# Values greater than 0 represent delayed payments (e.g., 1 = one month late).
display(
    df_fe["ANY_PAST_DUE"]
    .value_counts()
    .rename("CLIENTS")
)


ANY_PAST_DUE
0    19931
1    10069
Name: CLIENTS, dtype: int64